# Tratamento dos Datasets — Whirlpool Thermal Predictor

Este é o **segundo notebook** da pipeline de preparação de dados. Ele parte dos 7 datasets gerados pelo notebook anterior (`preparacao_datasets.ipynb`) e aplica os tratamentos definidos com o professor.

**Pré-requisito:** o notebook 1 deve ter sido executado antes, e os seguintes arquivos devem estar na mesma pasta:

- `dataset_mainback.csv`
- `dataset_ipc_comp.csv`
- `dataset_coil_temps.csv`
- `dataset_ct_knobs.csv`
- `dataset_main_vent_sys.csv`
- `dataset_oven_hmi.csv`
- `dataset_ipc_cooling.csv`
- `limites_ativos.csv`

> **Nota:** o `dataset_original.xlsx` nunca é modificado por nenhum notebook. O notebook 1 apenas lê dele. Este notebook 2 lê os CSVs gerados pelo notebook 1. Se algo der errado, basta rodar o notebook 1 de novo — os CSVs originais serão recriados.

### Tratamentos aplicados neste notebook

1. Excluir `Oven RTD Connector Body` do dataset Mainback (sensor com defeito).
2. Unificar `housing = "open - updated"` com `"open"`.
3. Excluir `AC Blower Inlet Max` e `AC Blower Inlet Avg` do Main Vent Sys (92,9% de nulos).
4. Reduzir a coluna losses ao perfil de perdas (nenhum, simone, miro, nao informado).
5. One-Hot Encoding nas colunas de configuração (com valores 0 e 1, não booleanos) e descarte da coluna run_idx.
6. Remover `Dedicated IPC/CKTP Outlet` de `limites_ativos.csv`.


## 1. Configuração e leitura dos datasets


In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2/colabs_atualizados"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import pandas as pd
import glob

# Lista de features de design (serao codificadas com One-Hot)
FEATURES = ["blower_direction", "housing", "board_position", "run_case",
            "losses", "deflector_hole", "material_coil_support"]

# Somente os 7 arquivos gerados pelo notebook 1. Qualquer arquivo derivado
# (_tratado deste notebook, _semnulos de notebooks posteriores) e ignorado:
# eles ja vem codificados e seriam lidos como se fossem base bruta.
DERIVADOS = ("_tratado", "_semnulos")
arquivos = sorted(f for f in glob.glob("dataset_*.csv")
                  if not any(s in f for s in DERIVADOS))
print("Datasets encontrados:", len(arquivos))

datasets = {}
for arq in arquivos:
    nome = arq.replace("dataset_", "").replace(".csv", "")
    datasets[nome] = pd.read_csv(arq)
    print(f"  {nome:16s} -> {datasets[nome].shape}")

faltando = [c for c in FEATURES if c not in datasets[arquivos[0]
            .replace("dataset_", "").replace(".csv", "")].columns]
assert not faltando, f"arquivos de entrada nao parecem ser a base bruta: falta {faltando}"

## 2. Excluir `Oven RTD Connector Body` do Mainback

Todas as 53 leituras válidas desse sensor estão entre 198,8 °C e 273,6 °C — consistentemente acima do limite de segurança de 150 °C. Esse padrão indica defeito de calibração, não temperatura real do componente.


In [ ]:
# Conferência antes de excluir
print("Estatísticas do Oven RTD Connector Body:")
print(datasets["mainback"]["Oven RTD Connector Body"].describe().round(2))
print()

# Exclusão da coluna
datasets["mainback"] = datasets["mainback"].drop(columns=["Oven RTD Connector Body"])
print("Coluna excluída. Novo shape do Mainback:", datasets["mainback"].shape)


## 3. Unificar `housing = "open - updated"` com `"open"`

Na análise prévia, controlamos a diferença de temperatura por `run_case` e o resultado mostrou que a diferença entre `"open"` e `"open - updated"` inverte de sinal dependendo do tipo de ensaio — sinal de que a diferença observada na média geral vem da distribuição desigual dos ensaios, não da categoria em si.


In [ ]:
for nome, df in datasets.items():
    n_antes = (df["housing"] == "open - updated").sum()
    if n_antes > 0:
        datasets[nome]["housing"] = df["housing"].replace({"open - updated": "open"})
        print(f"  {nome}: {n_antes} valores 'open - updated' -> 'open'")

# Conferência
print()
print("Categorias de housing após unificação:")
print(datasets["mainback"]["housing"].value_counts())


## 4. Componentes do mesmo tipo: avaliação da agregação

Foi avaliada a substituição dos 16 IGBTs por uma única coluna de média, do mesmo modo que havia sido feito com os Reso Caps. A agregação foi descartada em ambos os casos.

A justificativa inicial apoiava-se no desvio entre os membros do grupo, de cerca de 5 °C por ensaio. Esse valor, contudo, descreve a dispersão em torno do centro, e não a distância entre o membro mais quente e o mais frio. Medida essa amplitude, verifica-se que o IGBT mais quente supera o mais frio em 15,4 °C na média dos ensaios, chegando a 28,4 °C no pior caso.

A consequência é verificada contra o limite de segurança de 110 °C, comum aos 16 IGBTs. Em 13 dos 56 ensaios ao menos um IGBT atinge ou ultrapassa esse limite, enquanto a média do grupo acusa apenas 9. Quatro ensaios com estouro real tornam-se invisíveis sob a agregação. Nos Reso Caps, cujo limite é de 100 °C, o único ensaio em que um membro atinge o limite também é mascarado pela média.

Como a confiabilidade do alerta de proximidade do limite consta entre os critérios de sucesso definidos com a parceira, a agregação foi descartada e os componentes foram mantidos como colunas-alvo individuais. Cabe registrar que o número de colunas-alvo não disputa amostra com as features, uma vez que cada componente constitui um problema de regressão próprio.

A célula abaixo registra a verificação que sustenta a decisão.

In [ ]:
df = datasets["ipc_comp"]
limites_ref = pd.read_csv("limites_ativos.csv").set_index("tc_name")["limite_num"]

for prefixo in ["IGBT ", "Reso Cap ", "DCBUS Cap "]:
    cols = sorted([c for c in df.columns if c.startswith(prefixo)])
    if not cols:
        continue
    amplitude = df[cols].max(axis=1) - df[cols].min(axis=1)
    limite = limites_ref.reindex(cols).dropna()

    print(f"{prefixo.strip()} ({len(cols)} colunas)")
    print(f"  amplitude entre membros, por ensaio: media={amplitude.mean():.2f} C, maxima={amplitude.max():.2f} C")
    if len(limite):
        L = float(limite.iloc[0])
        algum = int((df[limite.index] >= L).any(axis=1).sum())
        pela_media = int((df[limite.index].mean(axis=1) >= L).sum())
        print(f"  limite de {L:.0f} C -> ensaios com algum membro no limite: {algum}; "
              f"detectados pela media: {pela_media}; mascarados: {algum - pela_media}")
    print()

n_alvos = len([c for c in df.columns if c not in FEATURES + ["run_idx"]])
print("Nenhuma coluna foi agregada. IPC Comp mantem", n_alvos, "colunas-alvo.")

## 5. Excluir colunas com excesso de nulos (Main Vent Sys)

`AC Blower Inlet Max` e `AC Blower Inlet Avg` têm **52 de 56 valores nulos** (92,9%). Com apenas 4 leituras cada, essas colunas não oferecem informação suficiente para o modelo aprender qualquer padrão e podem prejudicar a predição.


In [ ]:
df = datasets["main_vent_sys"]

# Conferência antes de excluir
for col in ["AC Blower Inlet Max", "AC Blower Inlet Avg"]:
    n_nulos = df[col].isna().sum()
    pct = round(100 * n_nulos / len(df), 1)
    print(f"  {col}: {n_nulos}/{len(df)} nulos ({pct}%)")

# Exclusão
df = df.drop(columns=["AC Blower Inlet Max", "AC Blower Inlet Avg"])
datasets["main_vent_sys"] = df
print(f"\nColunas excluídas. Novo shape do Main Vent Sys: {df.shape}")


## 6. Simplificação da coluna losses

A coluna losses registra o perfil de perdas térmicas aplicado no ensaio, mas o faz por meio de uma string que empacota até quatro informações distintas: o perfil propriamente dito (simone, miro ou nenhum), a ordem de montagem (front order ou back order), os setpoints utilizados (240/145/180/180, modified) e, em vários rótulos, uma repetição do próprio Run Case. Dessa sobreposição resultam 13 categorias para apenas 56 ensaios.

A verificação sobre os dados mostrou que a maior parte dessa cardinalidade não corresponde a informação nova. Em pyro, o perfil é sempre nenhum; em broil c1 e em broil c2, é sempre simone. Em três dos quatro tipos de ensaio, portanto, a coluna é inteiramente determinada pelo Run Case, que já figura como feature própria. A variação real ocorre apenas dentro de broil qualitative, em que 11 ensaios usam o perfil simone e 10 usam o perfil miro.

Por essa razão, a coluna foi reduzida ao perfil de perdas, com quatro níveis: nenhum, simone, miro e nao informado. A ordem de montagem e os setpoints foram descartados por falta de sustentação amostral, uma vez que a ordem back aparece em apenas 5 ensaios e os setpoints explícitos em outros 5. O prefixo com o Run Case foi descartado por redundância.

A simplificação reduz as colunas de entrada de 27 para 18. Além disso, os 6 ensaios sem registro de perdas passam a receber um nível próprio, em lugar de ficarem com todas as colunas de losses zeradas, situação que era ambígua para modelos lineares.

Cabe registrar uma limitação. Dentro de broil qualitative, o perfil miro ocorre em 10 de 10 ensaios com o deflector fechado, enquanto o perfil simone ocorre em 10 de 11 ensaios com o deflector aberto. As duas variáveis são, portanto, quase colineares nesta base, e a importância que o modelo vier a atribuir a cada uma não deve ser interpretada isoladamente. A separação exigiria ensaios que cruzassem os dois fatores.

In [ ]:
def perfil_de_perdas(valor):
    """Extrai apenas o perfil de perdas da string original."""
    if pd.isna(valor):
        return "nao informado"
    if valor == "no losses":
        return "nenhum"
    if "simone" in valor:
        return "simone"
    if "miro" in valor:
        return "miro"
    return "nao informado"


for nome, df in datasets.items():
    df["perfil_perdas"] = df["losses"].map(perfil_de_perdas)
    datasets[nome] = df.drop(columns=["losses"])

# A lista de features passa a usar o perfil no lugar da string original
FEATURES = ["perfil_perdas" if c == "losses" else c for c in FEATURES]
print("Features de design:", FEATURES)

# Conferencia: distribuicao do perfil e sua relacao com o run_case
ref = datasets["ipc_comp"]
print()
print("Perfis encontrados:")
print(ref["perfil_perdas"].value_counts().to_string())
print()
print("Perfil por run_case:")
print(pd.crosstab(ref["perfil_perdas"], ref["run_case"]).to_string())

## 7. One-Hot Encoding

Cada coluna de texto (feature categórica) é transformada em colunas de 0 e 1, uma para cada categoria existente.

A coluna run_idx é preservada no arquivo salvo, mas não é codificada nem deve entrar em X. Por ser identificador sequencial do ensaio, ela não descreve característica física alguma da configuração; por outro lado, é ela que permite, depois do treino, saber quais ensaios caíram no conjunto de teste e cruzar os sete datasets entre si, por exemplo para verificar se um ensaio anômalo o é em mais de uma área. A exclusão acontece no momento de montar X, e não na gravação do CSV.

Colunas que não são codificadas: run_idx, pelo motivo acima, e as colunas de componente, que são o alvo da predição.

Antes de codificar, é verificado se alguma feature possui valor nulo. Com dummy_na igual a False, um ensaio nessa situação receberia todas as colunas daquela feature zeradas, resultado indistinguível de uma categoria ausente e ambíguo para modelos lineares. Foi esse exatamente o problema que a Seção 6 resolveu para as perdas ao criar o nível nao informado.

Cabe registrar que versões recentes do pandas geram True e False em vez de 0 e 1 no get_dummies, de modo que a conversão para inteiro é feita explicitamente.

In [ ]:
# Verificacao previa: nenhuma feature pode chegar nula ao One-Hot
houve_nulo = False
for nome, df in datasets.items():
    faltantes = df[FEATURES].isna().sum()
    faltantes = faltantes[faltantes > 0]
    if not faltantes.empty:
        houve_nulo = True
        print(f"ATENCAO - nulos nas features de {nome}:")
        print(faltantes.to_string())
if not houve_nulo:
    print("Nulos nas features: nenhum")
print()

for nome, df in datasets.items():
    # Identifica as colunas de componente (alvo): tudo que nao e feature nem run_idx
    colunas_alvo = [c for c in df.columns if c not in FEATURES + ["run_idx"]]

    # Aplica One-Hot nas colunas de feature, convertendo para 0/1 inteiro
    dummies = pd.get_dummies(df[FEATURES], columns=FEATURES, dummy_na=False).astype(int)

    # Remonta o dataset: run_idx (rastreabilidade) + features codificadas + temperaturas.
    # run_idx e mantido no arquivo, mas nao deve compor X no momento do treino.
    datasets[nome] = pd.concat([df[["run_idx"]], dummies, df[colunas_alvo]], axis=1)

    print(f"  {nome:16s} -> {len(dummies.columns)} colunas de one-hot + "
          f"{len(colunas_alvo)} componentes (alvo) + run_idx")

### Conferência do One-Hot


In [ ]:
# Conferência 1: valores são 0 e 1, não True/False
df_check = datasets["mainback"]
oh_cols = [c for c in df_check.columns if c.startswith("blower_direction")]
print("Tipo das colunas de one-hot:", df_check[oh_cols].dtypes.unique().tolist())
print("Valores únicos:", sorted(df_check[oh_cols[0]].unique().tolist()))
print()

# Conferência 2: todo ensaio tem exatamente um perfil de perdas
perfil_cols = [c for c in df_check.columns if c.startswith("perfil_perdas_")]
soma_perfil = df_check[perfil_cols].sum(axis=1)
print(f"Colunas de perfil de perdas: {perfil_cols}")
print(f"Ensaios com exatamente um perfil marcado: {(soma_perfil == 1).sum()} de {len(df_check)}")
print()

# Conferência 3: housing sem 'open - updated'
housing_cols = [c for c in df_check.columns if c.startswith("housing_")]
print(f"Colunas de housing: {housing_cols}")


## 8. Ensaios com configuração idêntica

Após a codificação, diferentes ensaios podem apresentar exatamente o mesmo vetor de entrada. Isso ocorre porque a base contém repetições da mesma configuração de fogão, e o modelo não dispõe de nenhuma informação capaz de distinguir uma repetição da outra.

Duas consequências decorrem disso e precisam ser medidas antes do treino.

A primeira é que existe um teto de acurácia que nenhum modelo ultrapassa. Quando duas linhas têm entrada idêntica e alvos diferentes, a melhor predição possível é a média entre elas, e o erro residual é irredutível. A dispersão entre repetições fornece, portanto, uma estimativa do ruído experimental da base, contra a qual as métricas do modelo devem ser comparadas.

A segunda é que uma divisão aleatória por linha colocaria configurações idênticas em treino e em teste ao mesmo tempo, e o erro medido no teste sairia otimista. Como a meta acordada com a parceira é de margem de erro de até 10 por cento, medir errado neste ponto comprometeria a avaliação da entrega. A divisão deve, por isso, agrupar por configuração, e não por linha.

In [ ]:
# Chave de configuracao: o vetor de entrada que o modelo enxerga (sem run_idx e sem alvos)
ref = datasets["mainback"]
cols_entrada = [c for c in ref.columns if any(c.startswith(f + "_") for f in FEATURES)]
chave = ref[cols_entrada].astype(str).agg("|".join, axis=1)

contagem = chave.value_counts()
print(f"Ensaios: {len(ref)}")
print(f"Configuracoes distintas: {chave.nunique()}")
print(f"Configuracoes repetidas: {int((contagem > 1).sum())}, "
      f"cobrindo {int(contagem[contagem > 1].sum())} ensaios")
print()

# Ruido experimental: dispersao dos alvos entre ensaios de mesma configuracao
print("Desvio padrao medio entre repeticoes da mesma configuracao, por area:")
for nome, df in datasets.items():
    cols_e = [c for c in df.columns if any(c.startswith(f + "_") for f in FEATURES)]
    alvo = [c for c in df.columns if c not in cols_e + ["run_idx"]]
    k = df[cols_e].astype(str).agg("|".join, axis=1)
    repetidas = k.value_counts()
    repetidas = repetidas[repetidas > 1].index
    marcadas = k.isin(repetidas)
    ruido = df[marcadas].groupby(k[marcadas])[alvo].std().mean().mean()
    print(f"  {nome:16s} {ruido:5.2f} C")

## 9. Componentes sem nenhuma medição

O notebook 1 preserva, por decisão explícita, os componentes que não possuem medição em nenhum dos 56 ensaios: eles aparecem como coluna inteiramente nula. A alternativa seria deixá-los cair silenciosamente no pivot, o que faria qualquer componente futuro sem medição desaparecer da base sem registro.

A exclusão desses componentes é feita aqui, de forma registrada, tanto das colunas-alvo quanto da tabela de referência de limites. Um componente sem nenhuma leitura não pode ser aprendido nem avaliado, e a linha correspondente na tabela ficaria órfã na comparação pós-predição.

In [ ]:
# Colunas-alvo inteiramente nulas: componentes que nunca foram medidos
sem_medicao = []
for nome, df in datasets.items():
    alvo = [c for c in df.columns if c not in FEATURES + ["run_idx"]]
    vazias = [c for c in alvo if df[c].isna().all()]
    if vazias:
        datasets[nome] = df.drop(columns=vazias)
        sem_medicao.extend(vazias)
        print(f"  {nome}: removidas {len(vazias)} colunas sem medicao -> {vazias}")

if not sem_medicao:
    print("Nenhum componente sem medicao encontrado.")
print()

# A tabela de referencia acompanha a mesma exclusao
limites = pd.read_csv("limites_ativos.csv")
print("Tabela de limites antes:", len(limites), "componentes")
limites = limites[~limites["tc_name"].isin(sem_medicao)]
print("Tabela de limites depois:", len(limites), "componentes")

limites.to_csv("limites_ativos_tratado.csv", index=False)
print("Tabela salva: limites_ativos_tratado.csv")

## 10. Salvar os datasets tratados


In [ ]:
for nome, df in datasets.items():
    arquivo_saida = f"dataset_{nome}_tratado.csv"
    df.to_csv(arquivo_saida, index=False)
    print(f"  {arquivo_saida:40s} -> {df.shape}")

print()
print("Todos os datasets tratados foram salvos.")


## 11. Resumo das mudanças

| Tratamento | O que foi feito | Onde |
|---|---|---|
| Oven RTD Connector Body | Coluna excluída (sensor com defeito) | Mainback |
| Housing "open - updated" | Unificado com "open" | Todos os 7 datasets |
| AC Blower Inlet Max/Avg | Colunas excluídas (92,9% de nulos) | Main Vent Sys |
| losses | Reduzida ao perfil de perdas: 13 colunas viram 4 | Todos os 7 datasets |
| One-Hot Encoding | 0/1 em 7 colunas de feature, totalizando 18 colunas de entrada | Todos os 7 datasets |
| run_idx | Mantido no CSV para rastreabilidade; excluido de X no treino | Todos os 7 datasets |
| Losses nulo | 6 ensaios recebem o nivel "nao informado" | Todos os 7 datasets |
| Componentes sem medição | Coluna-alvo e linha da tabela de limites removidas | Todos os 7 datasets e limites_ativos |

### Próximas etapas (fora do escopo deste notebook)

- **Outliers:** sinalizar por IQR, por componente, sem agrupar por run_case.
- **Split treino/teste:** feito dentro de cada área, no momento do treino.
- **Active Limit:** usado na comparação pós-predição, depois de ter modelo funcional.
